# Figure 1 — Benchmark design and data landscape

What goes *into* the benchmark: the three data regimes, the datasets, where they sit on the confounding axis, the simulation grid, and the method inventory. No results.

**Each panel is defined and drawn in its own cell.** Edit the function, re-run that one cell, see the result. The last cell assembles all of them at the submission size (180 x 205 mm) and writes the PDF that goes to the journal.

Run the Setup, Shared helpers and Data cells first; after that the panel cells are independent of each other.

## Setup

In [ ]:
# Everything the panels read is relative to benchmark/, so make that the working directory
# whether the notebook was opened from benchmark/ or from benchmark/figures/.
if (basename(getwd()) == "figures") setwd("..")
stopifnot("run this from benchmark/ or benchmark/figures/" = file.exists("config/methods.tsv"))

# _panels.R holds what every figure shares: the method inventory and its ordering, the metric
# catalogue, the two normalisations used for ranking, the theme, and save_figure().
source("figures/_panels.R")
suppressMessages(library(ggrepel))

results_dir <- "results"

## Shared helpers

Small pieces more than one panel below needs.

In [ ]:
dataset_confounding_table <- function(datasets, results_dir)
  dataset_confounding(results_dir, sub("^results/data/", "", datasets$path))

# Panels b and c describe the same datasets, so they share one encoding (and, assembled, one key):
# colour says whether a set is scored or only used for calibration, shape says the assay.
role_scale <- function()
  scale_colour_manual(values = c(yes = track_colours()[["A"]], no = "grey55"), name = "role",
                      labels = c(yes = "evaluation", no = "calibration"),
                      guide = guide_legend(order = 1, override.aes = list(size = 1.8, alpha = 1)))

assay_scale <- function()
  scale_shape_manual(values = c(16, 17), name = "assay",
                     guide = guide_legend(order = 2, override.aes = list(size = 1.8, alpha = 1)))

## Data

In [ ]:
datasets <- read.delim("data/DATASETS.tsv")
datasets <- datasets[datasets$track %in% c("real", "mock"), ]
scenarios <- read.delim("config/scenarios.tsv")
methods <- method_table()
methods <- methods[methods$role != "sensitivity", ]
confounding <- dataset_confounding_table(datasets, results_dir)

cat(nrow(datasets), "datasets,", nrow(scenarios), "scenarios,", nrow(methods), "methods\n")

## a. Design schematic

Three data regimes feed one wrapper interface; four metric families read its output. The three rows share both edges, so every arrow is a straight drop.

In [ ]:
panel_design <- function() {
  # Each row divides the same 12-unit width evenly, so the rows line up at both edges whatever the
  # number of boxes in them.
  row <- function(level, kind, heads, bodies, gap = 0.25) {
    width <- (12 - gap * (length(heads) - 1)) / length(heads)
    left <- (seq_along(heads) - 1) * (width + gap)
    data.frame(xmin = left, xmax = left + width, x = left + width / 2, y = level, kind = kind,
               head = heads, body = bodies)
  }
  boxes <- rbind(
    row(3, "data", c("Curated real data", "Designed protocol runs", "Simulation"),
        c("10 sets, 7,115 samples\nproxy metrics only", "4 sets, known composition\nor repeated specimens",
          "213 scenarios\nfull ground truth")),
    row(2, "interface", "One wrapper interface",
        "counts + batch + phenotype   ->   corrected table or embedding"),
    row(1, "metrics", c("Batch removal", "Signal retention", "Downstream", "Practicality"),
        c("R2, ASW, LISI", "R2, cLISI, oracle", "DA, cross-study", "time, memory, failures")))
  half <- 0.36
  drops <- rbind(data.frame(x = boxes$x[boxes$kind == "data"], from = 3 - half, to = 2 + half),
                 data.frame(x = boxes$x[boxes$kind == "metrics"], from = 2 - half, to = 1 + half))
  ggplot(boxes) +
    geom_rect(aes(xmin = xmin, xmax = xmax, ymin = y - half, ymax = y + half, fill = kind)) +
    geom_text(aes(x, y + 0.15, label = head), size = 2.3, fontface = "bold") +
    geom_text(aes(x, y - 0.13, label = body), size = 1.9, lineheight = 0.95, colour = "grey20") +
    geom_segment(data = drops, aes(x = x, xend = x, y = from - 0.03, yend = to + 0.03),
                 arrow = arrow(length = unit(1, "mm"), type = "closed"), linewidth = 0.3,
                 colour = "grey40") +
    scale_fill_manual(values = c(data = "#dce9f5", interface = "#b9d3ea", metrics = "grey93"),
                      guide = "none") +
    coord_cartesian(xlim = c(0, 12), ylim = c(1 - half, 3 + half), expand = FALSE) +
    labs(title = "Three data regimes, one interface, four metric families") +
    theme_void(base_size = 7) +
    theme(plot.title = element_text(size = 7.5, face = "bold"), plot.title.position = "plot",
          plot.margin = margin(2, 2, 2, 2, "mm"))
}

show_panel(panel_design(), 180, 42)

## b. Dataset landscape

Size against number of batches. Point area is the number of taxa; shape is the assay; blue sets are scored, grey ones are kept for calibration. Panel c uses the same colour and shape, so the two share one key in the assembled figure.

In [ ]:
panel_landscape <- function(datasets) {
  ggplot(datasets, aes(samples, n_batch)) +
    geom_point(aes(size = features, shape = assay, colour = evaluated), alpha = 0.85) +
    geom_text_repel(aes(label = dataset), size = 1.9, colour = "grey15", min.segment.length = 0,
                    max.overlaps = Inf, force = 25, force_pull = 0.3, box.padding = 0.6,
                    point.padding = 0.3, max.iter = 100000, seed = 3,
                    segment.size = 0.2, segment.colour = "grey60") +
    scale_x_log10(expand = expansion(0.09)) +
    # extra room below: four of the datasets sit in one cluster near the bottom of the panel and
    # their labels have nowhere else to go
    scale_y_continuous(expand = expansion(mult = c(0.2, 0.09))) +
    scale_size_area(max_size = 4.5, breaks = c(10, 100, 700)) +
    role_scale() + assay_scale() +
    labs(x = "samples (log scale)", y = "batches", size = "taxa", title = "Dataset landscape") +
    guides(size = guide_legend(order = 3)) +
    theme_bench() + theme(legend.position = "bottom")
}

show_panel(panel_landscape(datasets), 95, 75)

## c. Confounding axis

Where each real dataset falls on Cramer's V. The top axis marks the five levels the simulation sweeps. mbqc at 0 is the fully crossed anchor.

In [ ]:
panel_confounding <- function(datasets, confounding, simulated_levels) {
  placed <- data.frame(dataset = datasets$dataset, v = confounding, assay = datasets$assay,
                       evaluated = datasets$evaluated)
  placed$dataset <- factor(placed$dataset, placed$dataset[order(placed$v)])
  ggplot(placed, aes(v, dataset)) +
    geom_vline(xintercept = simulated_levels, colour = "grey60", linetype = "22", linewidth = 0.25) +
    geom_segment(aes(x = 0, xend = v, yend = dataset), colour = "grey75", linewidth = 0.3) +
    geom_point(aes(colour = evaluated, shape = assay), size = 1.8) +
    role_scale() + assay_scale() +
    # the simulated levels get the top axis to themselves, so they never collide with the data
    scale_x_continuous(limits = c(0, 1.03), expand = expansion(0.01), breaks = seq(0, 1, 0.25),
                       sec.axis = dup_axis(breaks = simulated_levels, labels = format(simulated_levels),
                                           name = "levels swept in the simulation")) +
    labs(x = "batch-phenotype confounding (Cramer's V)", y = NULL,
         title = "Where the real datasets sit on the confounding axis") +
    theme_bench() +
    theme(panel.grid.major = element_blank(), legend.position = "bottom",
          axis.title.x.top = element_text(size = 6, colour = "grey35"),
          axis.text.x.top = element_text(colour = "grey35"),
          axis.line.x.top = element_blank(), axis.ticks.x.top = element_blank())
}

show_panel(panel_confounding(datasets, confounding, c(0, 0.3, 0.6, 0.9, 1)), 95, 75)

## d. Simulation parameter space

One strip per factor, one point per scenario: what the 213-scenario grid actually covers.

In [ ]:
panel_simulation_space <- function(scenarios) {
  axes <- c(conf = "confounding (V)", n_batch = "batches", n_per = "samples per batch",
            imbalance = "batch size ratio", bias_sd = "multiplicative bias (sd)",
            affected = "taxa hit by bias", scale_sd = "variance shift (sd)",
            dropout = "extra dropout", depth_fold = "depth ratio",
            da_prop = "DA taxa", da_log2fc = "DA effect (log2)")
  sampled <- bind_rows(lapply(names(axes), function(k)
    data.frame(parameter = axes[[k]], value = scenarios[[k]])))
  sampled$parameter <- factor(sampled$parameter, axes)
  ggplot(sampled, aes(value, 1)) +
    geom_point(size = 0.5, alpha = 0.3, colour = track_colours()[["A"]],
               position = position_jitter(height = 0.3, seed = 1)) +
    facet_wrap(~ parameter, ncol = 4, scales = "free_x") +
    scale_x_continuous(n.breaks = 4) +
    labs(x = NULL, y = NULL, title = "Simulation parameter space (213 scenarios)") +
    theme_bench() +
    theme(axis.text.y = element_blank(), axis.ticks.y = element_blank(), axis.line.y = element_blank(),
          panel.grid.major.y = element_blank(), panel.spacing.x = unit(3, "mm"))
}

show_panel(panel_simulation_space(scenarios), 110, 75)

## e. Method inventory

What each method needs and what it returns, as a dot matrix: a filled dot is yes, a hollow one is no.

In [ ]:
panel_inventory <- function(methods) {
  properties <- c(needs_phenotype = "needs\nphenotype", needs_controls = "needs\ncontrols",
                  binary_only = "binary\nonly", transductive = "sees all\nsamples")
  ordered <- rev(methods$label[match(method_order(methods), methods$method)])
  long <- bind_rows(lapply(names(properties), function(k)
    data.frame(label = factor(methods$label, ordered), property = properties[[k]],
               value = factor(methods[[k]], c("yes", "no", "na")))))
  returned <- data.frame(label = factor(methods$label, ordered), property = "returns",
                         value = methods$output)
  ggplot(mapping = aes(property, label)) +
    geom_point(data = long, aes(shape = value, colour = value, size = value)) +
    geom_text(data = returned, aes(label = value), size = 1.8, colour = "grey15") +
    scale_shape_manual(values = c(yes = 16, no = 1, na = 45), name = NULL,
                       labels = c(yes = "yes", no = "no", na = "not applicable")) +
    scale_colour_manual(values = c(yes = track_colours()[["A"]], no = "grey70", na = "grey50"),
                        name = NULL, labels = c(yes = "yes", no = "no", na = "not applicable")) +
    scale_size_manual(values = c(yes = 1.9, no = 1.3, na = 3), name = NULL,
                      labels = c(yes = "yes", no = "no", na = "not applicable")) +
    scale_x_discrete(limits = c(unname(properties), "returns"), position = "top") +
    labs(x = NULL, y = NULL, title = "Method inventory") +
    theme_bench() + theme_tiles() +
    theme(panel.grid.major.y = element_line(colour = "grey93", linewidth = 0.2),
          axis.text.x.top = element_text(lineheight = 0.9), legend.position = "bottom")
}

show_panel(panel_inventory(methods), 75, 75)

## Assembled figure

Everything above, in the layout that goes to the journal: 180 x 205 mm, one page.
The panels shrink here — that is the real submission size, and the font sizes above were
chosen for it. Panels b and c share one key; d and e share the bottom row.

In [ ]:
datasets_row <- (panel_landscape(datasets) |
                 panel_confounding(datasets, confounding, c(0, 0.3, 0.6, 0.9, 1))) +
  plot_layout(guides = "collect") & theme(legend.position = "bottom")
inventory_row <- (panel_simulation_space(scenarios) | panel_inventory(methods)) +
  plot_layout(widths = c(1.5, 1))

figure <- assemble(panel_design() / datasets_row / inventory_row +
                   plot_layout(heights = c(0.58, 1, 1.05)))

save_figure(figure, "results/figures/fig1.pdf", 180, 205)
show_panel(figure, 180, 205, scale = 1.5)